# Final submissions — Kaggle "perfect-fit"

This notebook is self-contained: given `../data/dataset.csv` and
`../data/test.csv`, it produces the four Kaggle submissions we kept at the
end of the competition.

| Output CSV | CV MAE | Public LB MAE | Description |
|---|---|---|---|
| `submission_ebm_heavy_smooth.csv` | 3.08 | **4.90** | Single EBM with heavy smoothing. Cleanest "one model" baseline. |
| `submission_ensemble_cross_LE.csv` | 2.97 | **2.94** | 0.5·LIN(no x9) + 0.5·EBM(no x4). Our best LB-confirmed submission. |
| `submission_ensemble_triple_locked_b_lambda50.csv` | 2.82 | untested | 0.25·LIN_b + 0.25·EBM(no x4) + 0.5·EBM(full). Integer-locked linear. |
| `submission_router_A1_triple.csv` | **1.84** | untested | Route "safe" rows to A1 closed form, others to the triple ensemble. |

The next cell is the full work report (a copy of `REPORT.md` at the repo
root), explaining what the data was, what generalised, and what didn't.
Skip past it if you just want to run the code.

**Runtime**: ~15 min end-to-end on a laptop. The 5-fold CV cell is the
slowest; you can skip it and jump straight to full-data training if
you trust the numbers above.


# The Perfect Fit — Work Report

A concise narrative of the Kaggle "Perfect Fit" competition: what the
data was, what generalised, what didn't, and why.

**Terminology** (used throughout):

- **MAE** — Mean Absolute Error. Our evaluation metric.
- **CV** — 5-fold Cross-Validation on `dataset.csv`, using
  `KFold(shuffle=True, random_state=42)`. We always compute out-of-fold
  predictions explicitly so we can slice them by subgroup.
- **LB** — Public Leaderboard (Kaggle).
- **DGP** — Data-Generating Process (the true underlying function from
  features to target).
- **EBM** — Explainable Boosting Machine (interpret-core's
  `ExplainableBoostingRegressor`): a gradient-boosted additive model
  with bounded shape functions per feature + selected 2-way interactions.
- **GAM** — Generalised Additive Model (`pygam`'s cubic-spline family).
- **Sentinel** — a magic value that encodes "missing". Here,
  `x5 = 999.0` on ~15% of rows.

## 1. Competition and data

- Tabular regression, metric **MAE**. Submission = `(id, target)` on
  1,500 test rows.
- Training: 1,500 rows in `dataset.csv`. Features
  `x1, x2, x4, x5, x6, x7, x8, x9, x10, x11` (the competition skips
  `x3`) plus categoricals `Country` and `City`. Target continuous,
  `std ≈ 24.1`.
- Test: 1,500 rows in `test.csv`, same schema without `target`.

## 2. Data quirks we exploited

- **`Country` is constant at `Spain`.** Drop it.
- **`City` is binary** (Zaragoza / Albacete). Encode as a 0/1 column;
  its coefficient is always ~24–25 (large effect).
- **`x5` has a hard sentinel at 999.0.** Non-sentinel values are
  Uniform(7, 12). Imputation strategies (mean, median, k-Nearest-
  Neighbours, gradient boosting) all hit the same `~1.25` per-row
  Mean-Absolute-Error on sentinel rows — the Uniform(7, 12) median-
  optimal bound. **Nothing beats median imputation for x5.**
- **`x6² + x7² = 18²` exactly** on every row. Only the angle
  `θ = atan2(x7, x6)` varies. θ is uniform on `[−π, π]` and
  independent of every other feature and of the sentinel indicator.
  x6 and x7 are noise.
- **`x4` is bimodal with a gap at `[−0.167, 0.167]`.** Training has
  zero observations in the gap; the test set has 508 rows (33.9%)
  inside it.

## 3. Training vs test distribution shift

The single most consequential finding. On `dataset.csv` vs `test.csv`:

- `corr(x4, x9) = +0.832` in training, **`+0.001` in test**.
- Training `(x4, x9)` forms two disjoint clusters
  (`x4>0 → x9 ~ N(5.97, 0.57)`, `x4<0 → x9 ~ N(4.02, 0.57)`).
  Within-cluster correlation is zero.
- Test draws `x4` and `x9` independently; 734 test rows (48.9%)
  land in the off-diagonal quadrants `{x4>0, x9<5}` and
  `{x4<0, x9>5}` that are empty in training.
- All other pairwise correlations are `< 0.06` on both sets.
- Sentinel rate is preserved (14.8% train, 15.2% test).

**Consequence.** Any model that learns an x4 ↔ x9 relationship — an
interaction term, a residualised `x9_resid = x9 − β·x4`, a cluster-
centered `x9_wc`, or an inflated linear `β_x9` — extrapolates wrongly
on the 49% of test rows in the empty quadrants. Density-ratio
reweighting cannot fix this: reweighting redistributes mass over
observed rows, and the off-diagonal quadrants have no observed rows.

## 4. Signal discovery

Marginal linear correlations with target (by decreasing magnitude):
`City, x4, x5, x8, x9, x10, x11`. Additional findings:

- **`x1` and `x2` carry only nonlinear signal.** x1 has an
  inverted-U shape (GAM R² 0.109); x2 oscillates, fit well by
  `cos(5π·x2)` (GAM R² 0.068).
- **`x10 × x11` is a genuine pairwise interaction** (pure-interaction
  Root-Mean-Square 3.02 after an additive baseline; noise floor
  ~1.71). No other pair exceeds the noise floor.
- **Causal-discovery tools (the PC algorithm and DirectLiNGAM —
  linear-acyclic causal-graph recovery methods) returned a false
  positive `x4 → x9` edge.** Both assume linear causal structure and
  no selection bias; the selection-biased training correlation
  satisfied their independence tests. The `0.001` test correlation
  falsifies the causal claim.

## 5. The A1 closed-form formula and its failure

A sibling branch reverse-engineered a zero-parameter closed-form
(we call it **A1**) that fits most training rows to machine precision:

```
target = −100·x1² + 10·cos(5π·x2)
       + 15·x4 + 20·𝟙(x4 > 0)
       − 8·x5_imp + 15·x8 − 4·x9
       + x10·x11 − 25·𝟙(City = Zaragoza) + 92.5
```

- Training CV: overall MAE 1.80; non-sentinel 0.38; sentinel 10.0.
- **93.3% of non-sentinel training rows have `|residual| < 0.01`.**
  The 6.7% imperfect rows all lie in the quadrant `x4 < 0 AND
  x8 < 0`, where residual ≈ `−18.4·x8` (std 0.76). We call this
  subset the "clamp".
- **Public LB: MAE 10.80** — ~6× worse than its CV.

Two distinct failure modes explain the CV-LB gap:

1. `+20·𝟙(x4>0)` is a hard step that misfires on the 508 test rows
   inside the x4 gap (training had zero observations there, so the
   step was unfalsifiable during reverse-engineering).
2. `−4·x9` extrapolates on the 734 off-diagonal test rows.

Together these cover ~83% of the test set.

## 6. Models and CV→LB behaviour

| Family | Variant | CV MAE | LB MAE |
|---|---|---|---|
| Linear | baseline (City + x4 + x5 + x8 + x10 + x11) | 9.98 | — |
| Linear | + splines(x1, x2) + x10·x11, no x9 | 3.70 | 7.38 |
| Linear | locked-integer skeleton + cluster-centered x9 | 2.90 | 10.75 |
| Closed form | A1 (hand-engineered, step at x4=0) | 1.80 | 10.80 |
| Closed form | A2 (linear least-squares on hand basis) | 3.49 | 9.44 |
| GAM | tuned + x10·x11 | 3.56 | — |
| LightGBM | tuned | 4.66 | — |
| EBM | default | 3.24 | — |
| EBM | tuned (smoothing 2,000 rounds) | 3.11 | 5.66 |
| EBM | heavy smoothing (smoothing 2,000 + refine 1,500) | 3.08 | **4.90** |
| EBM | all-smoothed (smoothing 4,000, no refine) | 3.03 | — |
| Ensemble | EBM + GAM, 70/30 weighted | 2.91 | 6.47 |
| Ensemble | **cross_LE = 0.5·LIN(no x9) + 0.5·EBM(no x4)** | 2.97 | **2.94** |
| Ensemble | triple = 0.25·LIN + 0.25·EBM(no x4) + 0.5·EBM(full) | 2.82 | — |
| Router | safe → A1; else → triple ensemble | **1.84** | — |

Column names follow the convention **LIN** = hand-crafted linear
design matrix (`x1², cos(5π·x2), x4, x5_imp, x5_is_sent, x8, x10,
x11, x10·x11, City`, optionally `x9`). **EBM(no x4)** and
**EBM(no x9)** are EBM on the full feature set with one column
removed; **EBM(full)** keeps both.

Three patterns in the table:

1. **Parametric models catastrophically miss on LB** (A1, A2, the
   locked-integer linear). Their closed-form x9 coefficient encodes
   training joint structure that doesn't exist in test.
2. **EBM's bounded shape functions generalise.** EBM cannot
   extrapolate past its training bins, which is a feature: on
   off-diagonal test rows, EBM holds the nearest-bin value rather
   than projecting a trend. CV-to-LB degradation is ~1.8× (best
   among non-ensemble models).
3. **Ensembles that mix complementary failure modes help
   dramatically.** `cross_LE` pairs a linear model with no x9
   (immune to the shift) against an EBM with no x4 (handles
   everything else nonparametrically). Their errors on off-diagonal
   rows partially cancel. **CV 2.97 → LB 2.94** is unusually tight.

## 7. Final submissions

Four CSV files live in `submissions/`. The notebook
`notebooks/final_submissions.ipynb` rebuilds them byte-identically.

| File | CV MAE | LB MAE |
|---|---|---|
| `submission_ebm_heavy_smooth.csv` | 3.08 | **4.90** |
| `submission_ensemble_cross_LE.csv` | 2.97 | **2.94** |
| `submission_ensemble_triple_locked_b_lambda50.csv` | 2.82 | untested |
| `submission_router_A1_triple.csv` | 1.84 | untested |

## 8. Noise floor and placement

- **Sentinel x5 irreducible noise floor.** x5 has slope ≈ −8 on
  target; imputing a Uniform(7, 12) feature with its median leaves
  ≈ 1.25 per-row MAE. Weighted by the 15.2% sentinel rate in test:
  `0.152 × 8 × 1.25 = 1.52` MAE. **No submission can beat 1.52 on
  the public LB.**
- **LB top cluster** sits at 1.65–1.71 (within 0.2 of the floor),
  implying the leaders recovered the DGP to ≈ 0.15 MAE on non-
  sentinel rows.
- **Our best LB (2.94)** is ~1.4 MAE above the top cluster; our
  best non-sentinel MAE on CV was 1.72 (cross_LE). The gap to
  the top is approximately "A1's 93% non-sentinel perfection on the
  safe quadrants" that we couldn't safely route to, not a
  modelling-quality gap.

## 9. Ideas we tried and rejected (one line each)

- **Drop x9 from EBM.** CV 3.83 vs 3.08 — hurts CV; x9 carries real
  within-training signal that EBM extracts without over-committing.
- **Residualise x9 against x4** (A2 variant). Encodes the spurious
  x4-x9 edge; LB 9.44 vs EBM's 5.66.
- **Within-cluster-centered x9** (`x9_wc`). Same issue expressed
  differently; LB 10.75.
- **Monotone EBM constraints** on x4, x5, x8, City. CV 3.49.
- **EBM with interactions disabled.** CV 4.16.
- **EBM forced linear on x4 via residualisation.** CV 3.33–3.61.
- **Per-cluster EBM** (one model per sign-of-x4). CV 3.50.
- **LightGBM tuned.** CV 4.66 — shallow trees + heavy regularisation
  needed to not overfit; still worse than default EBM.
- **GAM + x10·x11 interaction.** CV 3.56 — best additive-only
  model, still worse than EBM.
- **Random Forest.** CV ~8.8 — can't fit x1/x2 nonlinearity
  smoothly.
- **Quantile regression with splines.** CV 6.96 — L1 loss +
  high-dim spline basis → optimisation issues.
- **Huber regression.** CV 5.19 — no effect vs OLS; our outliers
  are sentinel-driven, not leverage-type.
- **k-Nearest-Neighbours (20 configs).** Best CV 9.33 — local
  similarity misses `cos(5π·x2)` structure.
- **NNLS** (Non-Negative Least-Squares) **stacking of EBM + GAM
  + A1.** CV 2.02 — A1 got 0.83 weight and memorised training;
  confirmed on LB by A1-alone's 10.80.
- **Density-ratio reweighting to break x4-x9 correlation**
  (classifier-based, KDE, Gaussian-copula). Weighted correlation
  dropped from 0.83 to 0.74 best; downstream CV moved ≤ 0.08.
  Structurally impossible to generate mass in the empty quadrants.
- **Synthetic off-diagonal augmentation** (resample x9, recompute
  target via A1). CV 4.40 — using A1 as DGP oracle backfires
  because A1's `−4·x9` is itself wrong on test.
- **A1 + EBM residual corrector.** CV 2.04 — EBM overfits the
  residual noise on the 93% of rows A1 already fits exactly.
- **Classifier-based clamp trigger for the router.** LightGBM
  reached AUC 0.76 on the 23% hidden "clamp" rows within the
  `x4<0, x8<0` quadrant; routing variants (soft blend, hard
  thresholds at 0.3/0.4/0.5/0.6) gave CV 1.85–1.86, worse than
  the parameter-free base router (CV 1.84). AUC would need to
  reach ~0.9 before the 4-to-1 reward-to-cost ratio of routing
  pays off.
- **x6/x7 angle θ as a feature.** KS test `p = 0.89` against the
  clamp indicator; independent of every other feature. Adding θ
  hurt EBM CV 3.47 → 3.56.
- **Integer-locked coefficient hypothesis for A2's basis.** CV 2.90
  (identical to free fit) — the DGP is integer, but the formula
  itself still extrapolates wrongly on test (LB 10.75).

## 10. Open questions

- **Hidden clamp trigger.** The 23% of `x4<0 AND x8<0` rows with
  `residual ≈ −18.4·x8` look like a Bernoulli(0.23) hidden binary
  in the DGP. No observed feature or two-way interaction predicts
  it above AUC 0.76. Three-way feature interactions, unsupervised
  structure discovery, or a non-tabular side channel might push it
  to AUC 0.9.
- **DGP oracle for test-distribution augmentation.** Synthetic
  off-diagonal rows would close the train-test shift if we had the
  true target function. A1 is not that function on the off-diagonal
  side; we don't know what is.


## 1. Setup


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from interpret.glassbox import ExplainableBoostingRegressor

SEED = 42
SENTINEL = 999.0  # the magic missing-value code on x5
N_SPLITS = 5

# Try both notebook-run-from-notebooks/ and notebook-run-from-repo-root:
HERE = Path.cwd()
REPO = HERE.parent if (HERE / "../data/dataset.csv").exists() else HERE
DATA = REPO / "data"
SUBS = REPO / "submissions"
SUBS.mkdir(exist_ok=True)

FEATURES_ALL = ["x1", "x2", "x4", "x5", "x8", "x9", "x10", "x11"]


## 2. Load data


In [2]:
train = pd.read_csv(DATA / "dataset.csv").reset_index(drop=True)
test  = pd.read_csv(DATA / "test.csv").reset_index(drop=True)
y     = train["target"].values
is_sent = (train["x5"] == SENTINEL).to_numpy()
print(f"train: {len(train)} rows  sentinels={is_sent.sum()}")
print(f"test : {len(test)} rows  sentinels={(test['x5']==SENTINEL).sum()}")


train: 1500 rows  sentinels=222
test : 1500 rows  sentinels=228


## 3. Shared preprocessing

Three helpers used across every model:

- **`preprocess(df, features, x5_median)`** — builds the EBM feature frame
  (x5 imputed with training median, adds `x5_is_sentinel` and binary
  `city` columns).
- **`design_matrix(df, x5_median, include_x4, include_x9)`** — builds
  the hand-crafted linear basis with `x1²`, `cos(5π·x2)`, `x10·x11`,
  and toggles for x4/x9 (used for the `LIN_x4` view).
- **`safe_mask(df)`** — the routing predicate. "Safe" means: x5 is not
  a sentinel, x4 is outside the training gap, and x9 matches its
  x4-cluster (true invariant of the DGP).


In [3]:
def preprocess(df: pd.DataFrame, features: list[str], x5_median: float) -> pd.DataFrame:
    out = df[features].copy()
    out["x5"] = out["x5"].where(out["x5"] != SENTINEL, x5_median)
    out["x5_is_sentinel"] = (df["x5"] == SENTINEL).astype(float)
    out["city"] = (df["City"] == "Zaragoza").astype(float)
    return out


def ebm_features(with_x4: bool, with_x9: bool) -> list[str]:
    feats = list(FEATURES_ALL)
    if not with_x4: feats.remove("x4")
    if not with_x9: feats.remove("x9")
    return feats


def design_matrix(df: pd.DataFrame, x5_median: float,
                  include_x4: bool, include_x9: bool) -> np.ndarray:
    is_sent = (df["x5"] == SENTINEL).astype(float).values
    x5 = df["x5"].where(df["x5"] != SENTINEL, x5_median).values
    city = (df["City"] == "Zaragoza").astype(float).values
    cols = [
        df["x1"].values ** 2,
        np.cos(5 * np.pi * df["x2"].values),
    ]
    if include_x4: cols.append(df["x4"].values)
    cols += [x5, is_sent, df["x8"].values, df["x10"].values, df["x11"].values,
             df["x10"].values * df["x11"].values, city]
    if include_x9: cols.append(df["x9"].values)
    return np.column_stack(cols)


X4_GAP_THRESH = 0.167  # empirical bimodal gap in x4


def safe_mask(df: pd.DataFrame) -> np.ndarray:
    x4 = df["x4"].to_numpy()
    x9 = df["x9"].to_numpy()
    sent = (df["x5"].to_numpy() == SENTINEL)
    x4_clear = np.abs(x4) > X4_GAP_THRESH
    cluster_match = ((x4 > 0) & (x9 > 5)) | ((x4 < 0) & (x9 < 5))
    return (~sent) & x4_clear & cluster_match


## 4. Model builders

Two EBM configurations:

- **`fit_ebm_heavy_smooth`** — `smoothing_rounds=2000, interaction_smoothing_rounds=500,
  max_rounds=2000`. Used for the single-model `ebm_heavy_smooth` submission
  (LB 4.90).
- **`fit_ebm_4k`** — `smoothing_rounds=4000, interaction_smoothing_rounds=1000,
  max_rounds=4000`. Used for the ensembles (`cross_LE`, triple, router).
  Slightly stronger smoothing → better CV (3.03 vs 3.08) at 2× training time.

The linear view uses `LinearRegression` on `design_matrix(...)` either
free-fit or pinned to the **integer-locked** coefficient set B that
matches the reverse-engineered A1/A2 formulas. The intercept is always
free so that the imputed-sentinel rows receive a correction.

`approach1_predict` is the A1 closed form — zero free parameters,
near-perfectly interpolates training (CV 1.80) but catastrophically
fails on the test set's x4-gap rows (LB 10.80). In the router we only
apply it where we have strong prior evidence it will work.


In [4]:
EBM_KW_HEAVY_SMOOTH = dict(
    interactions=10, max_rounds=2000, min_samples_leaf=10, max_bins=128,
    smoothing_rounds=2000, interaction_smoothing_rounds=500, random_state=SEED,
)
EBM_KW_4K = dict(
    interactions=10, max_rounds=4000, min_samples_leaf=10, max_bins=128,
    smoothing_rounds=4000, interaction_smoothing_rounds=1000, random_state=SEED,
)

def fit_ebm_heavy_smooth(X, y):
    return ExplainableBoostingRegressor(**EBM_KW_HEAVY_SMOOTH).fit(X, y)

def fit_ebm_4k(X, y):
    return ExplainableBoostingRegressor(**EBM_KW_4K).fit(X, y)


# Integer-locked coefs for LIN_x4 (A1/A2 declared integers).
# Column order must match design_matrix(include_x4=True, include_x9=False).
LOCKED_COEFS_B = {
    "x1^2": -100, "cos(5pi*x2)": +10, "x4": +30, "x5_imp": -8,
    "x5_is_sent": 0, "x8": +14, "x10": 0, "x11": 0,
    "x10*x11": +1, "city": -25,
}
LIN_COL_ORDER = ["x1^2", "cos(5pi*x2)", "x4", "x5_imp", "x5_is_sent",
                 "x8", "x10", "x11", "x10*x11", "city"]


def lin_x4_free(df_tr, df_va, x5_median):
    X_tr = design_matrix(df_tr, x5_median, include_x4=True, include_x9=False)
    X_va = design_matrix(df_va, x5_median, include_x4=True, include_x9=False)
    return LinearRegression().fit(X_tr, df_tr["target"].values).predict(X_va)


def lin_x4_locked_b(df_tr, df_va, x5_median):
    X_tr = design_matrix(df_tr, x5_median, include_x4=True, include_x9=False)
    X_va = design_matrix(df_va, x5_median, include_x4=True, include_x9=False)
    v = np.array([LOCKED_COEFS_B[c] for c in LIN_COL_ORDER])
    intercept = (df_tr["target"].values - X_tr @ v).mean()
    return X_va @ v + intercept


def approach1_predict(df: pd.DataFrame, x5_median: float) -> np.ndarray:
    """A1 closed form — reverse-engineered by a sibling branch.

    target = -100*x1^2 + 10*cos(5π*x2) + 15*x4 - 8*x5 + 15*x8
           - 4*x9 + x10*x11 - 25*is_zaragoza + 20*1(x4>0) + 92.5
    """
    x5 = df["x5"].where(df["x5"] != SENTINEL, x5_median).values
    is_zar = (df["City"] == "Zaragoza").astype(float).values
    x4_pos = (df["x4"].values > 0).astype(float)
    return (
        -100 * df["x1"].values ** 2
        + 10 * np.cos(5 * np.pi * df["x2"].values)
        + 15 * df["x4"].values
        - 8 * x5
        + 15 * df["x8"].values
        - 4 * df["x9"].values
        + df["x10"].values * df["x11"].values
        - 25 * is_zar
        + 20 * x4_pos
        + 92.5
    )


def mae(p, y):
    return float(np.mean(np.abs(p - y)))


## 5. 5-fold CV (optional — slow, ~8-10 min)

Reproduces the CV numbers quoted in the overview table. Set
`RUN_CV = False` to skip and jump straight to full-data training.


In [5]:
RUN_CV = True

if RUN_CV:
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    names = ["a1", "lin_x4_free", "lin_x4_b", "ebm_x9", "ebm_full", "ebm_heavy"]
    oof = {n: np.zeros(len(train)) for n in names}

    for fold, (tr, va) in enumerate(kf.split(train)):
        sub_tr = train.iloc[tr].reset_index(drop=True)
        sub_va = train.iloc[va].reset_index(drop=True)
        x5m = float(sub_tr.loc[sub_tr["x5"] != SENTINEL, "x5"].median())

        oof["a1"][va]          = approach1_predict(sub_va, x5m)
        oof["lin_x4_free"][va] = lin_x4_free(sub_tr, sub_va, x5m)
        oof["lin_x4_b"][va]    = lin_x4_locked_b(sub_tr, sub_va, x5m)

        feats = ebm_features(with_x4=False, with_x9=True)
        X_tr, X_va = preprocess(sub_tr, feats, x5m), preprocess(sub_va, feats, x5m)
        oof["ebm_x9"][va] = fit_ebm_4k(X_tr, sub_tr["target"].values).predict(X_va)

        feats = ebm_features(with_x4=True, with_x9=True)
        X_tr, X_va = preprocess(sub_tr, feats, x5m), preprocess(sub_va, feats, x5m)
        oof["ebm_full"][va] = fit_ebm_4k(X_tr, sub_tr["target"].values).predict(X_va)

        X_tr, X_va = preprocess(sub_tr, FEATURES_ALL, x5m), preprocess(sub_va, FEATURES_ALL, x5m)
        oof["ebm_heavy"][va] = fit_ebm_heavy_smooth(X_tr, sub_tr["target"].values).predict(X_va)

        print(f"  fold {fold+1}/{N_SPLITS} done")

    # Report
    triple  = 0.25 * oof["lin_x4_b"] + 0.25 * oof["ebm_x9"] + 0.50 * oof["ebm_full"]
    crossLE = 0.50 * oof["lin_x4_free"] + 0.50 * oof["ebm_x9"]
    routed  = np.where(safe_mask(train), oof["a1"], triple)

    print("\n{:<32s} {:>8s} {:>10s}".format("model", "overall", "non-sent"))
    print("-" * 54)
    for label, p in [
        ("A1 closed form",               oof["a1"]),
        ("LIN_x4 free",                  oof["lin_x4_free"]),
        ("LIN_x4 locked_b",              oof["lin_x4_b"]),
        ("EBM(no x4)",                   oof["ebm_x9"]),
        ("EBM(full)",                    oof["ebm_full"]),
        ("EBM heavy_smooth",             oof["ebm_heavy"]),
        ("cross_LE 0.5*(LIN_free+EBM9)", crossLE),
        ("triple 0.25+0.25+0.5",         triple),
        ("router(safe->A1, else->triple)", routed),
    ]:
        print(f"{label:<32s} {mae(p, y):8.3f} {mae(p[~is_sent], y[~is_sent]):10.3f}")


  fold 1/5 done


  fold 2/5 done


  fold 3/5 done


  fold 4/5 done


  fold 5/5 done

model                             overall   non-sent
------------------------------------------------------
A1 closed form                      1.800      0.376
LIN_x4 free                         3.695      2.537
LIN_x4 locked_b                     3.672      2.530
EBM(no x4)                          3.727      2.508
EBM(full)                           3.030      1.735
EBM heavy_smooth                    3.081      1.792
cross_LE 0.5*(LIN_free+EBM9)        2.973      1.718
triple 0.25+0.25+0.5                2.824      1.536
router(safe->A1, else->triple)      1.839      0.380


## 6. Full-data training

Every base model is refit on the full 1,500-row `dataset.csv`, and its
test predictions are cached in `preds`. This is the only step the
submission builders depend on — the CV section above is informational.


In [6]:
x5m = float(train.loc[train["x5"] != SENTINEL, "x5"].median())
preds: dict[str, np.ndarray] = {}

# A1 closed form
preds["a1"] = approach1_predict(test, x5m)

# Linear views — free-fit and integer-locked
X_tr = design_matrix(train, x5m, include_x4=True, include_x9=False)
X_te = design_matrix(test,  x5m, include_x4=True, include_x9=False)
preds["lin_x4_free"] = LinearRegression().fit(X_tr, y).predict(X_te)
v = np.array([LOCKED_COEFS_B[c] for c in LIN_COL_ORDER])
preds["lin_x4_b"] = X_te @ v + (y - X_tr @ v).mean()

# EBM views
feats = ebm_features(with_x4=False, with_x9=True)
preds["ebm_x9"] = fit_ebm_4k(
    preprocess(train, feats, x5m), y).predict(preprocess(test, feats, x5m))

feats = ebm_features(with_x4=True, with_x9=True)
preds["ebm_full"] = fit_ebm_4k(
    preprocess(train, feats, x5m), y).predict(preprocess(test, feats, x5m))

preds["ebm_heavy"] = fit_ebm_heavy_smooth(
    preprocess(train, FEATURES_ALL, x5m), y).predict(preprocess(test, FEATURES_ALL, x5m))

for k, p in preds.items():
    print(f"  {k:<14s}  mean={p.mean():+.3f}  std={p.std():.3f}  "
          f"range=[{p.min():+.2f}, {p.max():+.2f}]")


  a1              mean=-3.871  std=26.069  range=[-87.32, +71.95]
  lin_x4_free     mean=-4.022  std=23.204  range=[-79.07, +63.07]
  lin_x4_b        mean=-4.015  std=23.222  range=[-79.05, +62.89]
  ebm_x9          mean=-4.076  std=24.247  range=[-77.27, +75.15]
  ebm_full        mean=-3.979  std=22.957  range=[-77.61, +69.12]
  ebm_heavy       mean=-3.832  std=22.989  range=[-77.93, +69.82]


## 7. Build the four submissions

Each submission is a simple arithmetic combination of the cached `preds`.
See the formulas in the overview table at the top.


In [7]:
triple_b = 0.25 * preds["lin_x4_b"] + 0.25 * preds["ebm_x9"] + 0.50 * preds["ebm_full"]
cross_LE = 0.50 * preds["lin_x4_free"] + 0.50 * preds["ebm_x9"]
routed   = np.where(safe_mask(test), preds["a1"], triple_b)

outputs = {
    "submission_ebm_heavy_smooth.csv":                        preds["ebm_heavy"],
    "submission_ensemble_cross_LE.csv":                       cross_LE,
    "submission_ensemble_triple_locked_b_lambda50.csv":       triple_b,
    "submission_router_A1_triple.csv":                        routed,
}

for name, p in outputs.items():
    pd.DataFrame({"id": test["id"], "target": p}).to_csv(SUBS / name, index=False)
    print(f"  wrote {name}  mean={p.mean():+.3f}  range=[{p.min():+.2f}, {p.max():+.2f}]")


  wrote submission_ebm_heavy_smooth.csv  mean=-3.832  range=[-77.93, +69.82]
  wrote submission_ensemble_cross_LE.csv  mean=-4.049  range=[-76.36, +66.52]
  wrote submission_ensemble_triple_locked_b_lambda50.csv  mean=-4.012  range=[-76.99, +67.88]
  wrote submission_router_A1_triple.csv  mean=-4.061  range=[-80.30, +67.88]


## 8. Sanity checks

Validate each submission has the right shape, no NaNs, and a plausible
target distribution. If the existing Kaggle-confirmed CSVs are still
present, diff against them — we expect near-identical numbers for the
deterministic paths (`ebm_heavy_smooth`, `cross_LE`, `router`).


In [8]:
n_expected = len(test)
for name in outputs:
    df = pd.read_csv(SUBS / name)
    assert list(df.columns) == ["id", "target"], f"{name}: bad columns {df.columns}"
    assert len(df) == n_expected, f"{name}: length {len(df)} != {n_expected}"
    assert df["target"].notna().all(), f"{name}: NaN predictions"
    assert np.isfinite(df["target"]).all(), f"{name}: non-finite predictions"
    assert (df["id"].values == test["id"].values).all(), f"{name}: id mismatch"
    print(f"  {name}: OK  n={len(df)}  mean={df['target'].mean():+.3f}")


  submission_ebm_heavy_smooth.csv: OK  n=1500  mean=-3.832
  submission_ensemble_cross_LE.csv: OK  n=1500  mean=-4.049
  submission_ensemble_triple_locked_b_lambda50.csv: OK  n=1500  mean=-4.012
  submission_router_A1_triple.csv: OK  n=1500  mean=-4.061


## 9. What I would try next

See `LEARNINGS.md` for portable patterns (cross-view ensembling, router
pattern, sentinel-noise floor arithmetic). Two open threads in `CLAUDE.md`:

- **Hidden clamp trigger**. Inside the `x4<0 ∧ x8<0` quadrant, 23% of
  training rows have `residual ≈ -18.4·x8` relative to A1. A LightGBM
  classifier reaches AUC 0.76 but that isn't sharp enough to route on
  — false positives cost more than true positives gain. Better features
  (beyond x6/x7 angle, which is null) or a different parameterisation
  might push AUC past the 0.9+ threshold needed.
- **DGP oracle**. A1 + the clamp rule would get the top-4 LB cluster
  (1.65–1.71), but only if we can generate targets for off-diagonal
  (x4, x9) test rows — which requires the true DGP. Synthetic
  augmentation using A1 backfires because A1's x9 coefficient is itself
  wrong on test.
